# ME324 · Midterm assessment — A language model for Parliament

**Due Tuesday 18 August 2026 at 09:00 (Moodle) · 25% of the final mark · Individual work**

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/tsrobinson/me324/blob/main/assessment/midterm-starter.ipynb)

**How this notebook works.** This is an assessment, so unlike the labs there is no
Solutions section. Cells marked `# TODO` are yours to write; markdown cells marked
**✍️** are where your prose goes. Everything else is given and already works and is
provided to help you. You should complete this notebook in tandem with reading the 
brief I have supplied you.

**LLM usage is permitted.** We are assessing you in this new working environment. 
Just note you do need to fill in the **AI use appendix** at the bottom, and you 
must be able to explain every line you submit -- the teaching team may ask you to, 
in the week 3 labs.

## ⏱️ There are four parts to this notebook for which you should budget around 6 hours of time

- **Part A — Building a model (35 marks).** ~2–3 hours, much of it will be waiting for training runs.
- **Part B — Experimenting on your model (35 marks).** ~2–3 hours.
- **Part C — Discussing your model (25 marks).** ~1 hour.
- **Appendix — AI use (5 marks, with reporting).** ~15 minutes.

Do them in order!

## Run me first

In [ ]:
import math, os, time
import torch
import torch.nn as nn
from torch.nn import functional as F

torch.manual_seed(1337)          # reproducibility (same seed as every ME324 lab)

device = 'cuda' if torch.cuda.is_available() else 'cpu'
print('PyTorch', torch.__version__, '| device:', device)
if torch.cuda.is_available():
    print('GPU:', torch.cuda.get_device_name(0))
else:
    print('No GPU — Runtime > Change runtime type > GPU (everything still runs on CPU, just slower).')

## Section 1 · The data

Your corpus is **every attributed speech from five sitting days of the House of Commons** (9–16 July 2026), which equates to about 2.1 million characters. The format is always speaker name, colon, speech.

(The data was sourced via TheyWorkForYou.)

In [ ]:
!wget -nc https://raw.githubusercontent.com/tsrobinson/me324/main/assessment/data/hansard-2026.txt -O hansard.txt

with open('hansard.txt', 'r', encoding='utf-8') as f:
    text = f.read()

print('total characters:', len(text))
print('---- first 500 characters ----')
print(text[:500])

Have a scroll through the data before you model it. Note the register: procedural formulas ("I beg to move", "my hon. Friend"), long sentences, plenty of numbers and £ signs.

In [ ]:
chars = sorted(set(text))
print('vocab size:', len(chars))         # slightly different from our Shakespeare corpus
print(repr(''.join(chars)))

## Section 2 · Pipeline and baseline (given)

The pipeline comes from Lab 8. We have added one tracker that **records the loss history** and returns it, so you can plot your runs against each other. `count_params` is the parameter counter we will use to ensure you meet the constraints in the brief.

In [ ]:
def build_char_pipeline(text, block_size=8, batch_size=32, device="cpu", seed=1337):
    """Char-level tokenizer + batcher + loss estimator. (Lab 8, unchanged.)"""
    torch.manual_seed(seed)
    chars = sorted(set(text))
    vocab_size = len(chars)
    stoi = {ch: i for i, ch in enumerate(chars)}        # char  -> integer id
    itos = {i: ch for i, ch in enumerate(chars)}        # id    -> char
    encode = lambda s: [stoi[c] for c in s]             # string -> list[int]
    decode = lambda l: ''.join(itos[i] for i in l)      # list[int] -> string
    data = torch.tensor(encode(text), dtype=torch.long)
    n = int(0.9 * len(data))
    train_data, val_data = data[:n], data[n:]           # 90% train / 10% val

    def get_batch(split):
        d = train_data if split == 'train' else val_data
        ix = torch.randint(len(d) - block_size, (batch_size,))      # random starts
        x = torch.stack([d[i:i + block_size] for i in ix])          # (B, block_size)
        y = torch.stack([d[i + 1:i + block_size + 1] for i in ix])  # shifted by one
        return x.to(device), y.to(device)

    @torch.no_grad()
    def estimate_loss(model, eval_iters=200):
        out = {}
        model.eval()
        for split in ['train', 'val']:
            losses = torch.zeros(eval_iters)
            for k in range(eval_iters):
                xb, yb = get_batch(split)
                _, loss = model(xb, yb)               
                losses[k] = loss.item()
            out[split] = losses.mean().item()
        model.train()
        return out

    return dict(vocab_size=vocab_size, stoi=stoi, itos=itos, encode=encode,
                decode=decode, get_batch=get_batch, estimate_loss=estimate_loss,
                block_size=block_size, device=device)

In [ ]:
def train_model_tracked(model, get_batch, estimate_loss, max_iters=3000, eval_interval=500,
                        lr=1e-3, device="cpu"):
    """The labs' train_model, plus a record: returns (model, history), where history
    is a list of (iteration, train_loss, val_loss) tuples."""
    history = []
    optimizer = torch.optim.AdamW(model.parameters(), lr=lr)
    for it in range(max_iters):
        if it % eval_interval == 0 or it == max_iters - 1:
            losses = estimate_loss(model)
            history.append((it, losses['train'], losses['val']))
            print(f"step {it:5d} | train {losses['train']:.4f} | val {losses['val']:.4f}")
        xb, yb = get_batch('train')
        _, loss = model(xb, yb)                 # forward: model -> (logits, loss)
        optimizer.zero_grad(set_to_none=True)
        loss.backward()                         # backward: autograd
        optimizer.step()                        # update weights
    return model, history

In [ ]:
import matplotlib.pyplot as plt

def plot_histories(runs, title=""):
    """runs: dict of label -> history, e.g. {'block=8': h8, 'block=64': h64}."""
    plt.figure(figsize=(7, 4))
    for label, hist in runs.items():
        its = [h[0] for h in hist]
        plt.plot(its, [h[1] for h in hist], '--', alpha=0.5, label=f'{label} · train')
        plt.plot(its, [h[2] for h in hist], '-', label=f'{label} · val')
    plt.xlabel('iteration'); plt.ylabel('loss'); plt.title(title)
    plt.legend(); plt.grid(alpha=0.3); plt.show()

def count_params(model):
    """Every trainable parameter, biases included (house rule)."""
    return sum(p.numel() for p in model.parameters() if p.requires_grad)

### The bigram baseline

Here's Lab 8's bigram. Train it and uise the validation loss as the threshold for your Part A model to beat.

In [ ]:
class BigramLanguageModel(nn.Module):
    def __init__(self, vocab_size):
        super().__init__()
        # one row of next-char scores per character: no context, no memory.
        self.token_embedding_table = nn.Embedding(vocab_size, vocab_size)

    def forward(self, idx, targets=None):
        logits = self.token_embedding_table(idx)            # (B, T, vocab)
        loss = None
        if targets is not None:
            B, T, C = logits.shape
            loss = F.cross_entropy(logits.view(B * T, C), targets.view(B * T))
        return logits, loss

    @torch.no_grad()
    def generate(self, idx, max_new_tokens):
        for _ in range(max_new_tokens):
            logits, _ = self(idx)
            logits = logits[:, -1, :]                       # last position only
            probs = F.softmax(logits, dim=-1)
            idx_next = torch.multinomial(probs, num_samples=1)
            idx = torch.cat((idx, idx_next), dim=1)
        return idx

print('Bigram defined — same model as Lab 8.')

In [ ]:
P = build_char_pipeline(text, block_size=8, batch_size=32, device=device)

torch.manual_seed(1337)
bigram = BigramLanguageModel(P['vocab_size']).to(device)
print('parameters:', count_params(bigram))

bigram, bigram_history = train_model_tracked(bigram, P['get_batch'], P['estimate_loss'],
                                             max_iters=3000, eval_interval=500, lr=1e-2,
                                             device=device)
baseline = P['estimate_loss'](bigram)
print(f"\nBASELINE | val loss {baseline['val']:.4f} | per-char perplexity {math.exp(baseline['val']):.1f}")

In [ ]:
context = torch.zeros((1, 1), dtype=torch.long, device=device)   # start token = id 0
print(P['decode'](bigram.generate(context, max_new_tokens=400)[0].tolist()))

Notice that, untrained, the loss starts near ln(83) ≈ 4.42. Once trained, the baseline should land around **2.44** (perplexity ≈ 11.5 — the model is choosing among ~11 plausible next characters). If you are far from that, something upstream is wrong. The sample will be gibberish.

## Section 3 · Part A — Building a model (35 marks)

Design and train the best model you can within the rules.  Make sure to follow the docstring below, so the pipeline and training loop work unchanged and justify your design in ~150 words in the ✍️ cell.

You may reuse anything **you** built in Labs 5 and 8. Remember that once you pass the baseline as stipulated in the brief, a lower loss will not earn you extra marks.

In [ ]:
class MyLanguageModel(nn.Module):
    """Your model. The contract, same as every ME324 language model:

      forward(idx, targets=None) -> (logits, loss)
          idx: (B, T) integer ids; logits: (B, T, vocab_size);
          loss: cross-entropy over all positions if targets is given, else None.
      generate(idx, max_new_tokens) -> (B, T + max_new_tokens) integer ids

    Allowed ingredients: nn.Embedding, nn.Linear, activations, dropout, norm layers,
    nn.RNN / nn.GRU / nn.LSTM (or a recurrent cell of your own).

    Two practical constraints, because we rebuild your model from your checkpoint:
    keep this cell self-contained (everything the class needs is defined in it), and
    make your FINAL hyperparameters the constructor defaults, so that
    MyLanguageModel(vocab_size), with no other arguments, recreates the exact
    shapes you trained.
    """

    def __init__(self, vocab_size):
        super().__init__()
        # TODO: your architecture

    def forward(self, idx, targets=None):
        # TODO
        pass

    @torch.no_grad()
    def generate(self, idx, max_new_tokens):
        # TODO
        pass

In [ ]:
# Pipeline settings for your final model — change the arguments if you wish.
P = build_char_pipeline(text, block_size=8, batch_size=32, device=device)

torch.manual_seed(1337)
model = MyLanguageModel(P['vocab_size']).to(device)
n_params = count_params(model)
print(f'parameters: {n_params:,} / 500,000')
assert n_params <= 500_000, 'over budget - slim it down'

In [ ]:
# TODO: choose your training hyperparameters.
model, model_history = train_model_tracked(model, P['get_batch'], P['estimate_loss'],
                                           max_iters=3000, eval_interval=500, lr=1e-3,
                                           device=device)

In [ ]:
final = P['estimate_loss'](model)
print(f"yours    | train {final['train']:.4f} | val {final['val']:.4f} | perplexity {math.exp(final['val']):.1f}")
print(f"baseline | val {baseline['val']:.4f} | perplexity {math.exp(baseline['val']):.1f}")

plot_histories({'bigram': bigram_history, 'yours': model_history},
               title='Part A — your model vs the baseline')

context = torch.zeros((1, 1), dtype=torch.long, device=device)
print(P['decode'](model.generate(context, max_new_tokens=400)[0].tolist()))

### Save and verify your checkpoint

Your submission is the notebook **plus** the `.pt` file the next cell saves. The verify cell after it runs the same checks we run on every submission. **If the verify cell passes for you, it will pass for us.** Download the .pt file from Colab's file browser (folder icon, left sidebar) when you are done.

In [ ]:
# Save your checkpoint. Re-run this (and the verify cell) if you retrain your model.
bundle = dict(state_dict=model.state_dict(),
              block_size=P['block_size'],
              n_params=n_params,
              val_loss=final['val'])
torch.save(bundle, 'me324-midterm-checkpoint.pt')
print(f"saved me324-midterm-checkpoint.pt ({os.path.getsize('me324-midterm-checkpoint.pt')/1e6:.1f} MB)")

In [ ]:
# Verify your submission — the same checks we run on receipt. Leave this cell in.
b = torch.load('me324-midterm-checkpoint.pt', map_location='cpu', weights_only=True)

n = sum(t.numel() for t in b['state_dict'].values())
assert n <= 500_000, f'over budget: {n:,}'

banned = [k for k in b['state_dict']
          if any(s in k.lower() for s in ('attn', 'attention', 'in_proj', 'q_proj', 'k_proj', 'v_proj'))]
assert not banned, f'attention-like modules found: {banned}'

check = MyLanguageModel(P['vocab_size'])         # constructor defaults must rebuild your shapes
check.load_state_dict(b['state_dict'])
check.to(device)

P_check = build_char_pipeline(text, block_size=b['block_size'], batch_size=32, device=device)
torch.manual_seed(1337)
recomputed = P_check['estimate_loss'](check)['val']
print(f"recomputed val {recomputed:.4f} | reported {b['val_loss']:.4f} | baseline {baseline['val']:.4f}")
assert abs(recomputed - b['val_loss']) < 0.05, 'reported loss does not match the checkpoint'
assert recomputed <= baseline['val'] - 0.1, 'does not beat the baseline by 0.1'
print('all submission checks passed')

✍️ **Design justification (~150 words).** *Why this architecture, why these sizes, and what you spent your 500k parameters on. Your argument should be theoretical and specific. Replace this text with your answer.*

## Section 4 · Part B — Experimenting on your model (35 marks)

Pick **two** of the following hyperparameters and test them on your model, one at a time:

1. **Context length** — `block_size` (Lecture 8): a wider window conditions on more, but in a fixed-window model it costs parameters. Does the trade buy anything?
2. **Capacity** — hidden size, embedding size, or number of layers (depth vs width)
3. **Learning rate** — value or schedule (Lecture 4)
4. **Regularisation** — dropout or weight decay (Lecture 5)
5. **Batch size** (Lectures 4–5)
6. **Training length** — train much longer, and decide from the curves when you *should* have stopped (Lecture 5)

For each experiment, in this order:

- **State your hypothesis first**, in the ✍️ cell, *before* you run anything.
- **Run a fair comparison.**
- **Show the curves** with `plot_histories`.
- **Interpret** in the ✍️ cell: 150–250 words, referring to your actual numbers.

We will mark the reasoning and the fairness of the comparison, not whether the change helped.

### Experiment 1

✍️ **Hyperparameter:** *(name it)*

✍️ **Hypothesis (written before running):** *Replace this text with one or two sentences —
what will happen to the validation loss, and why.*

In [ ]:
# Experiment 1: change ONE thing between runs; keep everything else fixed.
#
# The pattern:
#   P_a = build_char_pipeline(text, block_size=..., batch_size=..., device=device)
#   torch.manual_seed(1337)
#   model_a = MyLanguageModel(P_a['vocab_size']).to(device)   # or a variant class
#   model_a, hist_a = train_model_tracked(model_a, P_a['get_batch'], P_a['estimate_loss'],
#                                         max_iters=..., lr=..., device=device)
#   ...then the same again with the one change...

# TODO: your runs

In [ ]:
# TODO: label your runs and plot them against each other.
# plot_histories({'setting A': hist_a, 'setting B': hist_b}, title='Experiment 1')

✍️ **Interpretation (150–250 words).** *What happened, was your hypothesis right, and
what in the curves tells you? Replace this text with your answer.*

### Experiment 2

✍️ **Hyperparameter:** *(name it)*

✍️ **Hypothesis (written before running):** *Replace this text with one or two sentences —
what will happen to the validation loss, and why.*

In [ ]:
# Experiment 2: change ONE thing between runs; keep everything else fixed.
#
# The pattern:
#   P_a = build_char_pipeline(text, block_size=..., batch_size=..., device=device)
#   torch.manual_seed(1337)
#   model_a = MyLanguageModel(P_a['vocab_size']).to(device)   # or a variant class
#   model_a, hist_a = train_model_tracked(model_a, P_a['get_batch'], P_a['estimate_loss'],
#                                         max_iters=..., lr=..., device=device)
#   ...then the same again with the one change...

# TODO: your runs

In [ ]:
# TODO: label your runs and plot them against each other.
# plot_histories({'setting A': hist_a, 'setting B': hist_b}, title='Experiment 2')

✍️ **Interpretation (150–250 words).** *What happened, was your hypothesis right, and
what in the curves tells you? Replace this text with your answer.*

## Section 5 · Part C — Discussing your model (25 marks)

Provide three short answers, ~200 words each, referencing your results — make sure to quote your own numbers and samples, not general facts about language models.

**C1 · Diagnose your final model** using the Lecture 5 taxonomy: underfitting,
overfitting, or about right? Argue from your train/validation curves. Then: what would
you try first with ten times the compute, and why that rather than something else?

✍️ *Replace this text with your answer (~200 words).*

**C2 · Take two or three generated samples.** What has the model genuinely learned about parliamentary English — structure, register, names, procedure? What does it consistently get wrong, and which limitation from the lectures explains that failure?

✍️ *Replace this text with your answer (~200 words).*

**C3 · The data.** Your training data is real MPs' words, scraped from the public record (published under the Open Parliament Licence, which permits this). Give one reason this use is clearly legitimate, and one genuine concern that would apply to training on scraped text in general.

✍️ *Replace this text with your answer (~200 words).*

## Appendix — AI use (5 marks, with reporting)

✍️ **Tools.** *Which AI assistants you used, and for what (writing code, debugging,
explaining, drafting prose). "None" is acceptable, but we don't expect it.*

✍️ **One correction.** *Paste one exchange where an assistant's suggestion was wrong, or
didn't fit the rules or the budget, and explain how you caught and fixed it. If you never
had to correct anything, pick one substantial piece of assistant-written code and explain
how you satisfied yourself it was right.*

✍️ **Declaration.** *"I can explain every line of this notebook." Leave this line in —
submitting the notebook asserts it.*

## Before you submit

- [ ] Runtime → **Restart and run all** completes without errors, in under ~20 minutes on a Colab GPU runtime
- [ ] All cell outputs visible and saved — do **not** clear outputs
- [ ] The parameter assert and the **verify cell** both pass on the fresh run
- [ ] Checkpoint downloaded from the Colab file browser
- [ ] Every ✍️ cell filled in (and both hypotheses genuinely written before the runs)
- [ ] Appendix complete
- [ ] **Two files**, renamed `ME324-midterm-<candidate-number>.ipynb` and `ME324-midterm-<candidate-number>.pt`, uploaded to Moodle by **09:00, Tuesday 18 August**